In [ ]:
import requests
import pandas as pd
import io
import re
from bs4 import BeautifulSoup

#Nível 1: Básico (URLs, Parâmetros e Cabeçalhos)

* Exercício 1.1: Faça uma requisição GET para a API pública do JSONPlaceholder ([https://jsonplaceholder.typicode.com/posts](https://jsonplaceholder.typicode.com/posts)) ou para a API do IBGE.
* Exercício 1.2: Utilize o argumento params para filtrar os resultados (por exemplo, buscando apenas o userId igual a 2 ou limitando a quantidade de resultados).
* Exercício 1.3: Na sua requisição, inclua um dicionário de cabeçalhos (headers) passando um User-Agent personalizado com o nome do seu projeto.
* Exercício 1.4: Imprima na tela o código de status HTTP (status_code) para confirmar o sucesso (código 200) e a URL final montada pelo requests (resposta.url). Lembre-se de sempre utilizar o parâmetro timeout (ex: timeout=10) para evitar que seu código fique travado indefinidamente.

In [ ]:
url_posts = "https://jsonplaceholder.typicode.com/posts"
params = {"userId": 2, "_limit": 3}
headers = {"User-Agent": "Projeto-WebDados/1.0"}

resposta = requests.get(url_posts, params=params, headers=headers, timeout=10)

print(f"Status HTTP: {resposta.status_code}")
print(f"URL final (com query string): {resposta.url}")

Status HTTP: 200
URL final (com query string): https://jsonplaceholder.typicode.com/posts?userId=2&_limit=3


#Nível 2: Intermediário (JSON, Erros e Arquivos Binários)


Exercício 2.1: Crie uma lista com três CEPs diferentes e faça um loop para consultá-los na API do ViaCEP ([https://viacep.com.br/ws/](https://viacep.com.br/ws/){cep}/json/). Utilize o método .json() para converter as respostas e armazene os resultados em um DataFrame do Pandas.

In [ ]:
ceps = ["01310100", "20040020", "90160000"]  # SP, RJ, RS
registros = []

for cep in ceps:
    url_cep = f"https://viacep.com.br/ws/{cep}/json/"
    r = requests.get(url_cep, timeout=10)
    r.raise_for_status()
    dados_cep = r.json()
    registros.append({
        "cep": dados_cep.get("cep"),
        "logradouro": dados_cep.get("logradouro"),
        "bairro": dados_cep.get("bairro"),
        "cidade": dados_cep.get("localidade"),
        "uf": dados_cep.get("uf")
    })

df_ceps = pd.DataFrame(registros)
df_ceps

,cep,logradouro,bairro,cidade,uf
0,01310-100,Avenida Paulista,Bela Vista,São Paulo,SP
1,20040-020,Praça Pio X,Centro,Rio de Janeiro,RJ
2,90160-000,Avenida da Azenha,Azenha,Porto Alegre,RS


Exercício 2.2: Escreva uma função de download segura utilizando um bloco try/except. Dentro dela, utilize resposta.raise_for_status() para capturar erros HTTP (como 404 ou 500) e imprima uma mensagem amigável caso a requisição falhe.

In [ ]:
def baixar_seguro(url, timeout=10):
    """Faz download com verificação de erros HTTP."""
    try:
        resposta = requests.get(url, timeout=timeout)
        resposta.raise_for_status()
        print(f"OK: {resposta.status_code} — {url}")
        return resposta
    except requests.exceptions.HTTPError:
        print(f"⚠️ Erro HTTP ao acessar {url} ({resposta.status_code})")
    except requests.exceptions.ConnectionError:
        print(f"⚠️ Não foi possível conectar em {url}")
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout ao acessar {url}")
    except requests.exceptions.RequestException:
        print(f"⚠️ Erro genérico ao acessar {url}")
    return None

ok = baixar_seguro("https://viacep.com.br/ws/01310100/json/")
falha = baixar_seguro("https://viacep.com.br/ws/99999999/json/")
nada = baixar_seguro("https://httpbin.org/status/404")
print("\nRetornos:", "ok" if ok else "None", "|", falha, "|", nada)

OK: 200 — https://viacep.com.br/ws/01310100/json/
OK: 200 — https://viacep.com.br/ws/99999999/json/
⚠️ Erro HTTP ao acessar https://httpbin.org/status/404 (404)

Retornos: ok | <Response [200]> | None


Exercício 2.3: Faça uma requisição para a URL [https://picsum.photos/400/400](https://picsum.photos/400/400) para baixar uma imagem aleatória. Salve o conteúdo bruto da resposta (acessado através de resposta.content) em um arquivo local com a extensão .jpg utilizando o modo de escrita em bytes ('wb').

In [ ]:
resposta_img = baixar_seguro("https://picsum.photos/400/400")

if resposta_img:
    with open("imagem_aleatoria.jpg", "wb") as arquivo:
        arquivo.write(resposta_img.content)
    print("Imagem salva: imagem_aleatoria.jpg")
    print("Tamanho (bytes):", len(resposta_img.content))
else:
    print("Download não realizado.")

OK: 200 — https://picsum.photos/400/400
Imagem salva: imagem_aleatoria.jpg
Tamanho (bytes): 19636


#Nível 3: Avançado (Webscraping e Ética)

Exercício 3.1: Antes de fazer a raspagem, faça uma requisição para o arquivo robots.txt do site que deseja acessar para verificar as permissões de acesso, demonstrando responsabilidade ética na coleta de dados.

In [ ]:
site = "https://books.toscrape.com"
robots = requests.get(f"{site}/robots.txt", timeout=10)
print(f"robots.txt status: {robots.status_code}")
print("-------------")
print(robots.text[:1000])

robots.txt status: 404
-------------
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.21.6</center>
</body>
</html>



Exercício 3.2: Acesse o site voltado para estudos de raspagem [https://books.toscrape.com/](https://books.toscrape.com/). Utilize a biblioteca BeautifulSoup com o analisador 'html.parser' para localizar os elementos da página. Extraia o título e o preço dos 5 primeiros livros utilizando métodos como .find() ou .select().

In [ ]:
resposta_site = requests.get(
    "https://books.toscrape.com/",
    headers={"User-Agent": "Projeto-WebDados/1.0, etica no scraping"},
    timeout=10
)
resposta_site.raise_for_status()

sopa = BeautifulSoup(resposta_site.text, "html.parser")

# Cada livro fica em
artigos = sopa.select("article.product_pod")[:5]

livros = []
for artigo in artigos:
    titulo = artigo.h3.a.get("title")  # atributo title tem o nome completo
    preco_raw = artigo.select_one("p.price_color").get_text()
    preco = re.sub(r"[^\d.,]", "", preco_raw)  # extrai só números (limpa moeda/símbolos)
    livros.append({"titulo": titulo, "preco": preco})

df_livros = pd.DataFrame(livros)
df_livros

,titulo,preco
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23


Exercício 3.3: Salve os dados extraídos dos livros em um arquivo CSV utilizando o Pandas.

In [ ]:
df_livros.to_csv("livros_toscrape.csv", index=False, encoding="utf-8")
print("CSV salvo: livros_toscrape.csv")
print(pd.read_csv("livros_toscrape.csv"))

CSV salvo: livros_toscrape.csv
                                  titulo  preco
0                   A Light in the Attic  51.77
1                     Tipping the Velvet  53.74
2                             Soumission  50.10
3                          Sharp Objects  47.82
4  Sapiens: A Brief History of Humankind  54.23


Exercício 3.4: Escolha uma página da Wikipedia que contenha uma tabela de dados (como listas de países ou populações). Utilize a função pandas.read_html() combinada com io.StringIO() para capturar a tabela da página e transformá-la diretamente em um DataFrame, sem a necessidade de usar o BeautifulSoup.

In [ ]:
url_wiki = "https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o"
html_wiki = requests.get(url_wiki, headers={"User-Agent": "Projeto-WebDados/1.0"}, timeout=15).text

# 1º) variação anual/pop
tabelas = pd.read_html(io.StringIO(html_wiki))
print("Tabelas encontradas:", len(tabelas))
df_wiki = tabelas[0]
df_wiki.head(10)